In [ ]:
import pytesseract  # Biblioteca para realizar OCR (Reconhecimento Óptico de Caracteres)
from PIL import Image  # Biblioteca para manipulação de imagens
from pdf2image import convert_from_path  # Função para converter páginas de um PDF em imagens

# Caminho do executável do Tesseract (necessário para realizar OCR)
pytesseract.pytesseract.tesseract_cmd = r'C:\Users\hfasa\AppData\Local\Programs\Tesseract-OCR\tesseract.exe'

# Caminho do arquivo PDF que será processado
pdf_path = r"C:\Users\hfasa\Desktop\Wares\DEV\Estudos\Cientista_de_dados\9. Combinação de modelos II\Material de estudo\Artigos para leitura\Stochastic GBM.pdf"

# Converter as páginas do PDF em imagens (uma imagem por página)
paginas = convert_from_path(pdf_path, dpi=300)

# Realizar OCR em cada página convertida
for i, imagem in enumerate(paginas):  # Itera sobre as imagens das páginas
    texto = pytesseract.image_to_string(imagem, lang='eng')  # Extrai o texto da imagem usando OCR
    print(f"Página {i+1}:")  # Exibe o número da página
    print(texto)  # Exibe o texto extraído da página

Página 1:
Stochastic Gradient Boosting

Jerome H. Friedman*

March 26, 1999

Abstract

Gradient boosting constructs additive regression models by sequentially fitting a simple
parameterized function (base learner) to current “pseudo”-residuals by least-squares at
each iteration. The pseudo-residuals are the gradient of the loss functional being minimized,
with respect to the model values at each training data point, evaluated at the current step. It
is shown that both the approximation accuracy and execution speed of gradient boosting can
be substantially improved by incorporating randomization into the procedure. Specifically,
at each iteration a subsample of the training data is drawn at random (without replacement)
from the full training data set. This randomly selected subsample is then used in place of the
full sample to fit the base learner and compute the model update for the current iteration.
This randomized approach also increases robustness against overcapacity of the base l

1. **Cite 5 diferenças entre o AdaBoost e o GBM**
2. **Acesse o link Scikit-learn - GBM, leia a explicação  (traduza se for preciso) e crie um jupyter notebook contendo o exemplo de classificação e de regressão do GBM.**
3. **Cite 5 hiperparâmetros importantes no GBM** 
4. **(Opcional) Utilize GridSearch para encontrar os melhores hiperparâmetros para o conjunto de dados do exemplo (load_iris)**
5. **Acessando o artigo do Jerome Friedman (Stochastic) e pensando no nome dado ao Stochastic GBM, qual é a maior diferença entre os dois algoritmos ?**

---

### **Atividade 1**

1. O modelo GBM não utiliza um stump como base, como acontece no AdaBoost. Em vez disso, o GBM parte de um valor inicial, que geralmente é a média dos valores da variável de interesse (no caso de regressão).

2. No GBM é utilizada uma floresta de árvores com maior profundidade, enquanto o AdaBoost costuma usar stumps (árvores rasas).

3. As respostas de cada submodelo no GBM não recebem pesos diferentes como no AdaBoost. No GBM, todas as árvores têm o mesmo peso e são apenas multiplicadas pelo `learning_rate`.

4. O cálculo do erro e a forma como os modelos fazem suas predições também são diferentes. O GBM tenta reduzir os resíduos do modelo anterior, enquanto o AdaBoost dá mais peso para os exemplos que foram classificados errado, tentando corrigi-los nas próximas iterações.

5. O GBM também permite escolher diferentes funções de perda, principalmente em problemas de regressão. No caso de árvores classificatórias, a função é fixa, mas para regressão existe uma variedade de funções disponíveis.

---

### **Atividade 2**

In [ ]:
# Importando bibliotecas essenciais
import numpy as np  # Biblioteca para manipulação de arrays e operações matemáticas
from sklearn.metrics import mean_squared_error  # Métrica para calcular o erro quadrático médio

# Importando datasets para exemplos de regressão e classificação
from sklearn.datasets import make_friedman1  # Dataset sintético para problemas de regressão
from sklearn.datasets import make_hastie_10_2  # Dataset sintético para problemas de classificação
from sklearn.datasets import load_iris  # Dataset clássico para classificação multiclasse

# Importando modelos de Gradient Boosting
from sklearn.ensemble import GradientBoostingRegressor  # Modelo para regressão
from sklearn.ensemble import GradientBoostingClassifier  # Modelo para classificação

from sklearn.model_selection import GridSearchCV  # Para busca em grade de hiperparâmetros
import pandas as pd

In [ ]:
X, y = make_hastie_10_2(random_state=0)
X_train, X_test = X[:2000], X[2000:]
y_train, y_test = y[:2000], y[2000:]

clf = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0,
    max_depth=1, random_state=0).fit(X_train, y_train)
clf.score(X_test, y_test)

In [ ]:
X, y = make_friedman1(n_samples=1200, random_state=0, noise=1.0)
X_train, X_test = X[:200], X[200:]
y_train, y_test = y[:200], y[200:]
est = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0,
    loss='squared_error'
).fit(X_train, y_train)
mean_squared_error(y_test, est.predict(X_test))

---

### **Atividade 3**

1. **`n_estimators` →** define o número de árvores (ou estágios) que serão treinadas sequencialmente para compor o modelo final. No caso de classificação binária, esse número corresponde diretamente à quantidade de árvores. Para tarefas de classificação multiclasse, o modelo treina `n_classes` árvores a cada iteração, então o total de árvores treinadas será `n_estimators` × `n_classes`.
2. **`learning_rate` →** é um hiperparâmetro essencial que controla o quanto cada nova árvore contribui para o modelo final. Ele atua como um "freio", reduzindo a influência de cada árvore individual no resultado, o que ajuda o modelo a generalizar melhor e evitar overfitting. Valores menores de `learning_rate` geralmente exigem um número maior de árvores (`n_estimators`) para alcançar um bom desempenho.
3. **`max_leaf_nodes` →** em vez de limitar a profundidade da árvore como o `max_depth`, esse parâmetro define o número máximo de folhas que cada árvore pode ter. Apesar de ser menos intuitivo de ajustar, ele costuma gerar resultados melhores porque força o modelo a focar em encontrar as divisões (splits) mais relevantes, em vez de apenas aprofundar a árvore até um certo ponto;
4. **`subsample` →** faz com que cada submodelo (árvore) seja treinado com apenas uma parte dos dados, e não com o conjunto todo. Isso ajuda o modelo a generalizar melhor, já que introduz uma aleatoriedade no processo. Essa amostragem é feita *sem reposição* e não usa pesos como no AdaBoost, então o modelo aprende com subconjuntos variados dos dados a cada etapa.
5. **`criterion` →** parâmetro que define qual métrica será usada para avaliar a qualidade das divisões nas árvores. Ele orienta o modelo sobre qual é o melhor ponto de quebra em cada split, influenciando diretamente a forma como as árvores aprendem com os dados.

---

### **Atividade 4**

In [ ]:
df = load_iris()
pd.DataFrame(df.data, columns=df.feature_names).head()

In [ ]:
params = {
    'n_estimators': [100, 200, 300, 400, 500],
    'learning_rate':[0.01, 0.1, 0.2, 0.3, 0.4],
    'subsample': [0.1, 0.2, 0.3, 0.4, 0.5],
}

In [ ]:
X, y = load_iris(return_X_y=True)

grid_clf = GridSearchCV(estimator= GradientBoostingClassifier(), param_grid = params)
grid_clf.fit(X, y)
# Acurácia do modelo com os melhores hiperparâmetros encontrados
grid_clf.score(X, y)

In [ ]:
print(grid_clf.best_params_)

---

### **Atividade 5**

O estudo de Stochastic GBM se aprofunda nas vantagens obtidas com o uso de amostragens sem reposição no treinamento de cada submodelo. O artigo demonstra testes com diferentes tamanhos de conjunto de dados, onde os maiores demonstraram maior ganho de performance com subamostras em torno de 60%, enquanto os menores devem se ater a 20% até 40%, árvores maiores precisam trabalhar com a randomização dos dados para evitar cair na armadilha do overfitting, comum nesses casos.